![](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# Cap 02 — Mensajes y LLM

profesor [Carlos Daniel Jiménez](danieljimenez88m@gmail.com)


En este capítulo conectamos LangGraph con un LLM real:
- `MessagesState`: estado especializado para conversaciones
- `add_messages`: reducer que acumula mensajes (no los reemplaza)
- Nodo LLM con `ChatOpenAI`
- Conversación multi-turn

**Dominio**: Libros de Stephen King  
**Flujo**: `START → node_experto_king → END`

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Falta OPENAI_API_KEY en .env"
print("API Key configurada ✓")

In [ ]:
import json
from typing import Annotated
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.graph.message import add_messages
from rich import print as rprint

## 1. El problema con TypedDict naívo

Si usamos un TypedDict simple para mensajes, cada nodo que devuelva `messages` **reemplazará** la lista completa.
Esto rompe la conversación multi-turn.

In [ ]:
from typing import TypedDict, List

class EstadoNaivo(TypedDict):
    messages: List  # ❌ Sin reducer: se reemplaza en cada nodo

# Demostración del problema
estado = {"messages": [HumanMessage(content="Hola")]}
# Si un nodo devuelve {"messages": [AIMessage(content="Hola!")]}
# ... el HumanMessage anterior desaparece

print("❌ Sin reducer: cada asignación borra el historial anterior")
print("✅ Con add_messages: los mensajes se acumulan")

## 2. MessagesState: la solución correcta

`MessagesState` ya tiene el reducer `add_messages` configurado.  
Cuando un nodo devuelve `{"messages": [nuevo_mensaje]}`, el reducer lo **añade** al historial.

In [ ]:
# MessagesState ya tiene: messages: Annotated[list[AnyMessage], add_messages]
# Equivalente manual:
class MiMessagesState(TypedDict):
    messages: Annotated[List, add_messages]

# O simplemente heredar de MessagesState:
# class MiEstado(MessagesState):
#     campo_extra: str

print("MessagesState gestiona automáticamente el historial de mensajes")
print("El reducer add_messages:")
print("  - Añade mensajes nuevos al historial")
print("  - Actualiza mensajes existentes por ID")
print("  - Nunca reemplaza mensajes anteriores")

## 3. Cargar datos de King y crear el LLM

In [ ]:
DATA_PATH = Path("../00_datos/king_books.json")
with open(DATA_PATH, encoding="utf-8") as f:
    king_books = json.load(f)

print(f"Cargados {len(king_books)} libros de King")

# Crear LLM
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-5-mini"), temperature=0)
print(f"LLM: {llm.model_name}")

## 4. Nodo LLM simple

In [ ]:
SYSTEM_PROMPT = """Eres un experto en la obra de Stephen King con 20 años de especialización literaria.
Responde siempre en español, con rigor académico pero accesible.
Cuando cites novelas, incluye el año de publicación y la época creativa del autor.
Relaciona los temas con el contexto cultural y biográfico de King cuando sea relevante.
Usa comparaciones con otros autores del género (Lovecraft, Koontz, Straub) para contextualizar.

Libros disponibles en tu base de conocimiento:
""" + ", ".join(b["titulo"] for b in king_books)

def node_experto_king(state: MessagesState) -> dict:
    """Nodo LLM: responde preguntas sobre Stephen King."""
    response = llm.invoke(
        [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    )
    return {"messages": [response]}

# Construir grafo simple
builder = StateGraph(MessagesState)
builder.add_node("experto_king", node_experto_king)
builder.add_edge(START, "experto_king")
builder.add_edge("experto_king", END)
graph = builder.compile()

print("Grafo compilado ✓")

## 5. Primera consulta

In [ ]:
resultado = graph.invoke({
    "messages": [HumanMessage(content="¿De qué trata 'It' de Stephen King?")]
})

print("=== Respuesta ===")
print(resultado["messages"][-1].content)

## 6. Conversación Multi-Turn

Con `MessagesState`, el historial se acumula automáticamente. Cada nueva invocación puede incluir los mensajes anteriores.

In [ ]:
# Primera vuelta
messages = [HumanMessage(content="¿Cuál es el tema central de The Shining?")]
resultado1 = graph.invoke({"messages": messages})
respuesta1 = resultado1["messages"][-1]
print("Pregunta 1:", messages[0].content)
print("Respuesta 1:", respuesta1.content[:200])

# Segunda vuelta — incluir historial completo
messages_v2 = resultado1["messages"] + [
    HumanMessage(content="¿Y cómo se relaciona eso con Pet Sematary?")
]
resultado2 = graph.invoke({"messages": messages_v2})
print("\nPregunta 2: ¿Y cómo se relaciona eso con Pet Sematary?")
print("Respuesta 2:", resultado2["messages"][-1].content[:300])

## 7. Demostrar que add_messages acumula

Vemos que al invocar con historial, el LLM tiene contexto de la conversación anterior.

In [ ]:
print(f"\nTotal mensajes en resultado2: {len(resultado2['messages'])}")
for i, msg in enumerate(resultado2["messages"]):
    tipo = type(msg).__name__
    print(f"  [{i}] {tipo}: {str(msg.content)[:80]}...")

## Resumen del Capítulo

| Concepto | Descripción |
|---------|-------------|
| `MessagesState` | Estado con reducer `add_messages` incorporado |
| `add_messages` | Reducer que acumula mensajes sin reemplazar |
| Nodo LLM | Función que invoca `llm.invoke([SystemMessage] + messages)` |
| Multi-turn | Pasar historial completo en cada invocación |

**Próximo capítulo**: Herramientas con `@tool` y el loop ReAct